# GeoRA 實作第一步：初始化一個真實 Q 矩陣

**這次要完成什麼？**

把 `Qwen2.5-1.5B-Instruct` 第 0 層的 Q 矩陣改寫成 GeoRA 的形式，
驗證初始化後的有效權重與這一層的輸出仍接近原始值。

流程：原始權重 → 兩個 mask → `W_geo` → rank-16 因子 → 凍結殘差 → 檢查輸出。

模型：`Qwen/Qwen2.5-1.5B-Instruct`。
下載時固定的 revision：`989aa7980e4cf806f80c7fef2b1adb7bc71aa306`。

使用這個倉庫 `.venv` 的 Python kernel。矩陣分解、forward 與誤差統計
統一在 CPU、FP64 上進行。
只讀一個權重矩陣和它的 bias，不載入整個模型。
第一次先讀第 1 節，再按順序看每個公式與程式碼。

原有版本已執行過；此副本清空輸出，請在新環境從頭逐格執行。
這次沒有進行 GRPO，也沒有驗證整個模型的 logits。

依據：[GeoRA v4 §3.1](https://arxiv.org/html/2601.09361v4#S3.SS1)、
[§3.2](https://arxiv.org/html/2601.09361v4#S3.SS2) 和 Appendix C。
以主文式 (2)–(8) 的乘法順序與縮放為準。


## 1. 取出原始 Q 權重

這層原本的計算是 $y=W_{\rm pre}x+b$。
`W_pre` 的尺寸是 **輸出維度 × 輸入維度 = 1536 × 1536**。
`bias` 是長度 1536 的向量，後面兩條計算路徑都加上同一個 bias。

`safe_open` 打開權重檔案；`get_tensor` 只取指定名字的張量。
原始權重保存為 BF16，我們用 `.double()` 轉為 FP64 做這次檢查。
精度轉換不會恢復保存時已經捨去的數字。

設定與論文一致：rank `r=16`、`alpha=32`、每個 mask 的選取比例 `rho=0.2`。
forward 的縮放係數是 $c=\alpha/r=2$。


In [ ]:
from pathlib import Path

import torch
from safetensors import safe_open

torch.set_num_threads(4)

project_directory = Path.cwd().resolve()
assert (project_directory / "geora_layers.py").is_file(), "Run from the repository root."
checkpoint_path = project_directory / "checkpoints/geora_base/model.safetensors"
tensor_name = "model.layers.0.self_attn.q_proj.weight"
bias_name = "model.layers.0.self_attn.q_proj.bias"

with safe_open(checkpoint_path, framework="pt", device="cpu") as checkpoint:
    stored_weight = checkpoint.get_tensor(tensor_name)
    bias = checkpoint.get_tensor(bias_name).double()

print("Stored weight:", stored_weight.shape, stored_weight.dtype)

W_pre = stored_weight.double()

r = 16
alpha = 32
rho = 0.2
c = alpha / r

assert W_pre.shape == (1536, 1536)
assert bias.shape == (1536,)
assert torch.isfinite(W_pre).all()

print("Working weight:", W_pre.shape, W_pre.dtype, W_pre.device)
print("Bias:", bias.shape)
print(f"r={r}, alpha={alpha}, rho={rho}, c={c}")


## 2. 第一個 mask：看原始權重的 rank-16 近似

先對 **原始 `W_pre`** 做 SVD：

$$W_{\rm pre}=U_{\rm pre}\Sigma_{\rm pre}V_{\rm pre}^{\top}.$$

`torch.linalg.svd` 的第三個輸出直接是 $V^\top$，所以叫 `Vh_pre`。
取最大的 16 個奇異值及相應方向，得到：

$$\widehat W_{{\rm pre},16}=U_{{\rm pre},16}\Sigma_{{\rm pre},16}V_{{\rm pre},16}^{\top}.$$

接著看這個近似矩陣的**元素絕對值**，標出最小約 20% 的位置，形成 `M_spec`。
這一步不是選最小的奇異值。

`reshape(-1)` 把所有元素排成一列，用來求 20% 分位數；
`<=` 產生一個同尺寸的布林矩陣：符合條件的位置為 `True`。
量化保存的權重可能有相同數值，選取比例因此不一定剛好是 20%。
本實作用 `torch.quantile` 的線性插值及論文的 `<=` 條件。


In [ ]:
U_pre, singular_values_pre, Vh_pre = torch.linalg.svd(
    W_pre,
    full_matrices=False,
)

Sigma_pre_r = torch.diag(singular_values_pre[:r])
W_pre_rank_r = U_pre[:, :r] @ Sigma_pre_r @ Vh_pre[:r, :]

spec_magnitudes = W_pre_rank_r.abs()
threshold_spec = torch.quantile(
    spec_magnitudes.reshape(-1),
    rho,
    interpolation="linear",
)
M_spec = spec_magnitudes <= threshold_spec

print("U_pre[:, :r]:", U_pre[:, :r].shape)
print("singular_values_pre = ", singular_values_pre)
print("Sigma_pre_r:", Sigma_pre_r.shape)

print("Vh_pre[:r, :]:", Vh_pre[:r, :].shape)
print("Spectral threshold:", threshold_spec.item())
print(f"Spectral mask selected: {M_spec.double().mean().item():.4%}")
print(f"M_spec = {M_spec}")




## 3. 第二個 mask，然後取兩者聯集

`M_euc` 直接看**原始 `W_pre` 的元素絕對值**，也標出最小約 20% 的位置。
兩個 mask 都是 1536 × 1536，並行產生。

`M_geo = M_spec | M_euc` 是聯集：任一 mask 選中的位置都留下。
因此聯集的比例通常大於 20%，不能把它稱為最後只保留 20% 的矩陣。

最後：

$$W_{\rm geo}=W_{\rm pre}\odot M_{\rm geo}.$$

**留下的是原始 `W_pre` 的值。** `W_pre_rank_r` 只用來決定 spectral mask 的位置。
PyTorch 相乘時把 `True` 當作 1、`False` 當作 0。


In [ ]:
euc_magnitudes = W_pre.abs()
threshold_euc = torch.quantile(
    euc_magnitudes.reshape(-1),
    rho,
    interpolation="linear",
)
M_euc = euc_magnitudes <= threshold_euc

M_geo = M_spec | M_euc
W_geo = W_pre * M_geo

assert M_geo.dtype == torch.bool
assert torch.equal(W_geo[M_geo], W_pre[M_geo])
assert torch.count_nonzero(W_geo[~M_geo]).item() == 0

print("Euclidean threshold:", threshold_euc.item())
print(f"Spectral mask selected: {M_spec.double().mean().item():.4%}")
print(f"Euclidean mask selected: {M_euc.double().mean().item():.4%}")
print(f"Union mask selected: {M_geo.double().mean().item():.4%}")
print("W_geo:", W_geo.shape)


## 4. 第二次 SVD：把 W_geo 寫成 rank-16 的兩個因子

現在分解的是 **`W_geo`**，不是上一節的 `W_pre`：

$$W_{\rm geo}=U_{\rm geo}\Sigma_{\rm geo}V_{\rm geo}^{\top}.$$

只保留前 16 個成分，將奇異值的平方根分到兩邊：

$$B_0=U_{{\rm geo},16}\Sigma_{{\rm geo},16}^{1/2},\qquad
A_0=\Sigma_{{\rm geo},16}^{1/2}V_{{\rm geo},16}^{\top}.$$

- `B0`：1536 × 16。
- `A0`：16 × 1536。
- `B0 @ A0`：1536 × 1536，是 `W_geo` 的 rank-16 近似。

`B0 @ A0` 不需要完全等於 `W_geo`。
下方利用 SVD 的性質檢查近似誤差：它的 Frobenius norm
應接近被捨去奇異值的平方和開根號。
本次 SVD、因子與 norm 統計都使用 FP64。
保留能量描述這個 **W_geo** 的近似品質，尚不代表真實 RLVR 更新的能量。


In [ ]:
U_geo, singular_values_geo, Vh_geo = torch.linalg.svd(
    W_geo,
    full_matrices=False,
)

Sigma_geo_r_sqrt = torch.diag(torch.sqrt(singular_values_geo[:r]))
B0 = U_geo[:, :r] @ Sigma_geo_r_sqrt
A0 = Sigma_geo_r_sqrt @ Vh_geo[:r, :]
W_geo_rank_r = B0 @ A0

approximation_error = torch.linalg.matrix_norm(
    W_geo - W_geo_rank_r,
)
discarded_singular_value_norm = torch.linalg.vector_norm(
    singular_values_geo[r:],
)
retained_energy = (
    singular_values_geo[:r].square().sum()
    / singular_values_geo.square().sum()
)

assert B0.shape == (W_pre.shape[0], r)
assert A0.shape == (r, W_pre.shape[1])
torch.testing.assert_close(
    approximation_error,
    discarded_singular_value_norm,
    rtol=1e-12,
    atol=1e-12,
)

print("B0:", B0.shape)
print("A0:", A0.shape)
print(f"Rank-{r} retained energy of W_geo: {retained_energy.item():.4%}")
print("Approximation error:", approximation_error.item())
print("Discarded singular-value norm:", discarded_singular_value_norm.item())


## 5. 建立凍結殘差，驗證有效權重

我們的 forward 將使用 $F+cBA$，其中 $c=\alpha/r=2$。
所以凍結部分必須先扣掉同樣的 $cB_0A_0$：

$$F=W_{\rm pre}-cB_0A_0.$$

`A`、`B` 是準備訓練的參數，初始值為 `A0`、`B0`。
`F.detach()` 不參與求導；`torch.nn.Parameter` 預設參與求導。
本 notebook 只準備參數，還不進行更新。

初始化時：

$$W_{{\rm eff},0}=F+cB_0A_0=W_{\rm pre}.$$

檢查的相對誤差是 $\|W_{{\rm eff},0}-W_{\rm pre}\|_F/\|W_{\rm pre}\|_F$。
FP64 仍有捨入誤差，所以檢查接近而不是逐位相同。

**這個抵消式本身不能證明 mask 寫對了**：任意因子都能被殘差抵消。
因此我們也分別檢查了 mask 留下的值、因子尺寸和 SVD 近似誤差。


In [ ]:
A = torch.nn.Parameter(A0.clone())
B = torch.nn.Parameter(B0.clone())
F = (W_pre - c * W_geo_rank_r).detach()

with torch.no_grad():
    W_eff_0 = F + c * (B @ A)
    weight_difference = W_eff_0 - W_pre
    relative_weight_error = (
        torch.linalg.matrix_norm(weight_difference)
        / torch.linalg.matrix_norm(W_pre)
    )
    maximum_weight_error = weight_difference.abs().max()

assert not F.requires_grad
assert A.requires_grad and B.requires_grad
assert relative_weight_error.item() < 1e-12

print("F:", F.shape, "requires_grad =", F.requires_grad)
print("A:", A.shape, "requires_grad =", A.requires_grad)
print("B:", B.shape, "requires_grad =", B.requires_grad)
print(f"Relative weight error: {relative_weight_error.item():.8e}")
print(f"Maximum absolute weight error: {maximum_weight_error.item():.8e}")


## 6. 用同一批輸入，檢查兩條 forward 路徑

權重相近後，再實際比較兩條不同的計算路徑。

原始路徑：$W_{\rm pre}x+b$。
GeoRA 路徑：$Fx+cB(Ax)+b$。

PyTorch 通常把一批輸入按**行**存放，這裡 `x` 是 4 × 1536。
所以程式裡等價地寫成：

$$Y_{\rm pre}=XW_{\rm pre}^{\top}+b,$$
$$Y_{\rm geo}=XF^{\top}+c(XA^{\top})B^{\top}+b.$$

`x @ A.T` 是 4 × 16，再乘 `B.T` 回到 4 × 1536。
相同的 bias 各加一次。
這不是先算出 `W_eff_0` 再乘輸入，因而也能檢查分支乘法與轉置是否一致。


In [ ]:
random_generator = torch.Generator(device="cpu")
random_generator.manual_seed(42)
x = torch.randn(
    4,
    W_pre.shape[1],
    generator=random_generator,
    dtype=torch.float64,
)

with torch.no_grad():
    output_pre = x @ W_pre.T + bias
    frozen_output = x @ F.T
    compressed_input = x @ A.T
    adapter_output = compressed_input @ B.T
    output_geo = frozen_output + c * adapter_output + bias

    output_difference = output_geo - output_pre
    relative_output_error = (
        torch.linalg.vector_norm(output_difference)
        / torch.linalg.vector_norm(output_pre)
    )
    maximum_output_error = output_difference.abs().max()

assert relative_output_error.item() < 1e-12

print("Input batch:", x.shape)
print("Compressed input:", compressed_input.shape)
print("Output:", output_geo.shape)
print(f"Relative output error: {relative_output_error.item():.8e}")
print(f"Maximum absolute output error: {maximum_output_error.item():.8e}")
print("Single-layer FP64 initialization checks passed.")


## 到這裡完成了什麼？

已經從真實權重依序建立兩個 mask、`W_geo`、rank-16 的 `B0` 與 `A0`，
再形成凍結殘差 `F`。
檢查通過表示：**這個 Q 層的 GeoRA 初始化，在 FP64 下保持原始有效權重與輸出。**

它尚不表示整模型初始化正確，也沒有證明 GeoRA 的訓練效果。
後續要先檢查 BF16 的數值誤差，再把同樣流程接到全部目標層。

### 下一步之前的自檢

1. 第一次 SVD 與第二次 SVD，各自分解哪個矩陣？
2. spectral mask 決定位置後，留下的是哪個矩陣的元素？
3. `B0 @ A0` 近似哪個矩陣？
4. 為什麼凍結部分要扣 `c * B0 @ A0`，而不是只扣 `B0 @ A0`？

<details><summary>答案</summary>

1. 第一次分解 `W_pre`，第二次分解 `W_geo`。
2. 留下原始 `W_pre` 的元素。
3. 近似 `W_geo`。
4. 因為 forward 加回的是 `c * B @ A`，初始化必須用相同係數抵消。

</details>


## 7. 下一步：凍結 F，只讓 A、B 訓練一步

前面確認了「開始訓練前，輸出不變」。現在要確認：**給出一個訓練目標後，A、B 能不能改變輸出，而 F 保持不變？**

仍然使用前面的 4 個輸入 `x`，每個長度 1536。
`output_pre` 是原始 Q 層對這 4 個輸入的輸出，形狀為 `(4, 1536)`。
我們人工指定一個同形狀的目標輸出：

$$\text{target\_output}=\text{output\_pre}+\text{target\_offset}.$$

`target_offset` 是固定亂數，每個元素來自均值 0、標準差 0.1 的常態分布。
目標一旦產生就固定，訓練只改模型的預測。
這不是文字資料，也不是 GRPO 的 reward；它只用來檢查這個 Q 層的梯度與參數更新。

每次從本節第一個程式格開始重跑，都會把 A、B 重設到初始化，方便重現同一個檢查。

In [ ]:
# Reset only the trainable factors; keep the original frozen residual.
A = torch.nn.Parameter(A0.clone())
B = torch.nn.Parameter(B0.clone())

target_generator = torch.Generator(device="cpu")
target_generator.manual_seed(43)
target_offset = 0.1 * torch.randn(
    output_pre.shape,
    generator=target_generator,
    dtype=torch.float64,
)
target_output = (output_pre + target_offset).detach()

# Save the values before the update for an exact comparison afterward.
F_before_step = F.clone()
A_before_step = A.detach().clone()
B_before_step = B.detach().clone()
with torch.no_grad():
    W_eff_before_step = F + c * (B @ A)

# SGD is used only for this diagnostic, not as the paper's optimizer.
learning_rate = 0.1
optimizer = torch.optim.SGD([A, B], lr=learning_rate)

assert not F.requires_grad
assert not target_output.requires_grad
print("Fixed target shape:", target_output.shape)
print("Trainable parameters: A and B")
print("Diagnostic learning rate:", learning_rate)

### 7.1 計算誤差，再求 A、B 的梯度

模型的預測仍然是前面的同一條 forward 路徑：

$$\text{predicted\_output}=xF^\top+c(xA^\top)B^\top+\text{bias}.$$

我們要讓預測接近固定目標，所以用均方誤差（MSE）作為這次檢查的 loss：

$$\text{loss}=\frac{1}{4\times1536}
\sum_{n=1}^{4}\sum_{j=1}^{1536}
(\text{predicted\_output}_{nj}-\text{target\_output}_{nj})^2.$$

`loss.backward()` 透過這條計算路徑求出 `A.grad`、`B.grad`。
這一步只計算梯度，還沒更新參數。
forward 不能包在 `torch.no_grad()` 裡，否則 PyTorch 不會記錄反向傳播需要的計算。

In [ ]:
optimizer.zero_grad()
predicted_output = x @ F.T + c * ((x @ A.T) @ B.T) + bias
loss_before = (predicted_output - target_output).square().mean()
loss_before.backward()

A_gradient_norm = torch.linalg.vector_norm(A.grad)
B_gradient_norm = torch.linalg.vector_norm(B.grad)

assert F.grad is None
assert torch.isfinite(A.grad).all()
assert torch.isfinite(B.grad).all()
assert A_gradient_norm.item() > 0
assert B_gradient_norm.item() > 0
assert torch.equal(A.detach(), A_before_step)
assert torch.equal(B.detach(), B_before_step)

print(f"Loss before step: {loss_before.item():.10f}")
print(f"A gradient norm: {A_gradient_norm.item():.10f}")
print(f"B gradient norm: {B_gradient_norm.item():.10f}")
print("F.grad:", F.grad)
print("Backward finished; parameter values have not changed yet.")

### 7.2 更新一次，檢查真正改了什麼

`optimizer.step()` 做這兩個更新，學習率就是前面設定的 `learning_rate`：

$$A_{\mathrm{new}}=A_{\mathrm{before}}-\text{learning\_rate}\,A.\mathrm{grad},$$
$$B_{\mathrm{new}}=B_{\mathrm{before}}-\text{learning\_rate}\,B.\mathrm{grad}.$$

F 沒有放進 optimizer，也不需要梯度，因此保持固定。
有效權重在這一步的變化為：

$$\begin{aligned}
\Delta W
&=(F+cB_{\mathrm{new}}A_{\mathrm{new}})
 -(F+cB_0A_0)\\
&=c(B_{\mathrm{new}}A_{\mathrm{new}}-B_0A_0).
\end{aligned}$$

注意：**訓練後不要重新用新的 A、B 計算 F。**
若每次都令 `F = W_pre - c * B @ A`，加回適配器後又會得到 `W_pre`，剛學到的更新就被抵消了。

In [ ]:
optimizer.step()

with torch.no_grad():
    output_after_step = x @ F.T + c * ((x @ A.T) @ B.T) + bias
    loss_after = (output_after_step - target_output).square().mean()

    F_change = torch.linalg.vector_norm(F - F_before_step)
    A_change = torch.linalg.vector_norm(A - A_before_step)
    B_change = torch.linalg.vector_norm(B - B_before_step)

    W_eff_after_step = F + c * (B @ A)
    delta_W_step = W_eff_after_step - W_eff_before_step
    delta_W_from_factors = c * (B @ A - B0 @ A0)
    weight_change_norm = torch.linalg.matrix_norm(delta_W_step)

assert torch.equal(F, F_before_step)
assert A_change.item() > 0
assert B_change.item() > 0
assert weight_change_norm.item() > 0
torch.testing.assert_close(
    delta_W_step,
    delta_W_from_factors,
    rtol=1e-12,
    atol=1e-14,
)
# This decrease is checked for this fixed target and step size only.
assert loss_after.item() < loss_before.item()

print(f"F change norm: {F_change.item():.10f}")
print(f"A change norm: {A_change.item():.10f}")
print(f"B change norm: {B_change.item():.10f}")
print(f"Effective weight change norm: {weight_change_norm.item():.10f}")
print(f"Loss: {loss_before.item():.10f} -> {loss_after.item():.10f}")
print("Single-step FP64 adapter training checks passed.")

### 這一步的結論，以及下一個關卡

檢查通過表示：這個真實 Q 矩陣的 **A、B 有梯度、能被更新，F 數值完全不變**，
有效權重確實因此改變；本次人工目標的誤差也下降了。
這個結果驗證單層的訓練連接，還不能用來比較 GeoRA 與 LoRA 的實際效果。

下一個關卡是把同樣的初始化接到完整模型的目標線性層，檢查初始化前後的 logits，
以及儲存／重新載入後是否仍然一致；完成後再接 GRPO。

**本節跑完後 A、B 已經改變。** 要再次檢查初始化，從第 5 節重跑；
要再次檢查同一個訓練步，從第 7 節第一個程式格重跑。